# K-fold Cross-validation for Semantic Segmentation

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/kfold_cross_validation.ipynb)

This notebook demonstrates k-fold U-Net training with
`geoai.train_segmentation_model`, performance plots with
`geoai.plot_performance_metrics`, and ensemble inference with
`geoai.semantic_segmentation_batch`. It uses the same training and visualization
functions as the [semantic segmentation example](train_segmentation_model.ipynb).

Each model trains on K-1 folds and validates on the remaining fold. K determines
the held-out fraction (approximately 1/K). The validation fold selects the best
checkpoint, controls early stopping, and supplies out-of-fold (OOF) predictions.
These are CV validation metrics; use an independent test dataset for final
performance assessment.


## Install packages

Use a Python environment with GeoAI, PyTorch, and segmentation-models-pytorch
installed. For this contribution, install the local GeoAI checkout: from the
repository root, run `python -m pip install -e .`. The commented notebook command
below assumes this notebook is opened from `docs/examples`; adjust the checkout
path if needed. Select the same environment as the notebook kernel and restart
it after installation. The import cell checks that k-fold support is available.


In [ ]:
# %pip install -e ../..

## Import libraries

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import inspect
import json
import logging
import uuid

import geoai
import numpy as np
import rasterio
import torch

if "k_folds" not in inspect.signature(geoai.train_segmentation_model).parameters:
    raise RuntimeError(
        "Install the GeoAI checkout with k-fold support, then restart the kernel."
    )
logging.basicConfig(level=logging.INFO, format="%(message)s")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("GeoAI:", geoai.__version__)
print("Device:", device)


## Prepare training data

Set the paths to your paired image and mask directories below. This example
expects byte RGB images and binary 0/1 masks with matching filename stems.
To create tiles from a raster and vector labels, follow the **Create training
data** section of the [semantic segmentation example](train_segmentation_model.ipynb).

Five folds train five models for up to 20 epochs each. For an execution check,
use a small paired dataset and fewer epochs. ImageNet encoder weights may
download on the first run.

In [ ]:
# Set these paths to your paired images and masks.
data_dir = Path("data/segmentation")
images_dir = data_dir / "images"
masks_dir = data_dir / "masks"

K = 5
MONITOR = "val_iou"  # or "val_loss"
EARLY_STOPPING_PATIENCE = None  # set to 5 to stop after 5 epochs without improvement
NUM_EPOCHS = 20
BATCH_SIZE = 4  # reduce if GPU memory is insufficient
TARGET_SIZE = None  # preserve native training tile sizes
ENCODER_NAME = "resnet34"
ENCODER_WEIGHTS = "imagenet"
ONLY_FOLD = None  # or a zero-based fold index
FOLD_GROUPS = None  # optional identifiers in sorted image file order

run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
)
output_dir = Path("output/kfold_segmentation") / run_id
print("New experiment directory:", output_dir)

# Use a new imagery directory for deployment; this default is a visualization demo.
ENSEMBLE_INPUT_DIR = images_dir

SEED = 42


## Train semantic segmentation model

Add `k_folds=K` to the usual training call. This example uses U-Net with a
ResNet34 encoder, ImageNet weights, RGB input, and two output classes.
GeoAI trains on K-1 folds and uses the remaining fold for validation,
checkpoint selection, early stopping, and OOF evaluation. K determines its
held-out fraction; val_split is ignored and no extra holdout is created.

Split recording, held-out batch inference, and OOF scoring happen inside GeoAI.
Each fold writes standard checkpoints and histories in `fold_<i>/`.
Without `k_folds`, the existing single-split behavior is preserved.

With `verbose=True`, each fold prints its training and validation/OOF
tile counts, filenames, and zero-based dataset indices before training.
Per-fold OOF metrics print after held-out inference. Full image/mask paths
for both sets are saved in that fold's `split.json`.


In [ ]:
summary = geoai.train_segmentation_model(
    images_dir=str(images_dir),
    labels_dir=str(masks_dir),
    output_dir=str(output_dir),
    architecture="unet",
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    num_channels=3,
    num_classes=2,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    seed=SEED,
    learning_rate=0.001,
    save_best_only=False,
    verbose=True,
    plot_curves=True,
    device=device,
    num_workers=0,
    target_size=TARGET_SIZE,
    k_folds=K,
    monitor=MONITOR,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    fold_groups=FOLD_GROUPS,
    only_fold=ONLY_FOLD,
)
print("K-fold training completed:", output_dir)


## Model performance analysis

Let's examine the training curves and model performance:

Use `geoai.plot_performance_metrics` on each fold's standard history file.
These curves use the held-out fold for checkpoint selection. OOF metrics score
that same fold after loading its best checkpoint; binary `class_1` is the foreground class.
Training-history metrics average over classes, so their values are
not directly interchangeable with foreground OOF metrics.

In [ ]:
print("Checkpoint criterion:", summary["monitor"])
print(json.dumps(summary["oof"], indent=2))
print("Complete:", summary["complete"], "folds:", summary["folds_completed"])
for fold in summary["per_fold"]:
    geoai.plot_performance_metrics(
        history_path=str(Path(fold["output_dir"]) / "training_history.pth"),
        figsize=(15, 5),
        verbose=True,
    )


## Out-of-fold predictions

Built-in training already called `geoai.semantic_segmentation_batch` on each
fold's held-out tiles. Collect those saved predictions below for the OOF audit
and comparison plot. The ensemble inference section later uses all selected
checkpoints together for deployment; it does not replace the OOF evaluation.


In [ ]:
mask_by_stem = {
    p.stem: p
    for p in masks_dir.iterdir()
    if p.suffix.lower() in (".tif", ".tiff", ".png", ".jpg", ".jpeg")
}
prediction_lookup = {}
for fold in summary["per_fold"]:
    fold_dir = Path(fold["output_dir"])
    split = json.loads((fold_dir / "split.json").read_text(encoding="utf-8"))
    predictions_dir = Path(fold["prediction_dir"])
    for image in split["held_out_files"]:
        path = Path(image)
        extension = ".tif" if path.suffix.lower() in (".tif", ".tiff") else ".png"
        prediction_lookup[path] = predictions_dir / f"{path.stem}_mask{extension}"
print("Held-out predictions:", len(prediction_lookup))


## Check saved predictions against OOF metrics (optional)

This optional audit reloads the predicted masks and confirms coverage and
pooled counts. `kfold_manifest.json` stores the fold assignments and
`kfold_summary.json` stores the returned metrics. In binary mode those metrics
are under `oof.micro.class_1` and `oof.macro.class_1`.

In [ ]:
from geoai.cross_validation import update_counts
from PIL import Image

recounted = {}
for image, prediction_path in prediction_lookup.items():
    with rasterio.open(prediction_path) as src:
        prediction = src.read(1)
    mask_path = mask_by_stem[image.stem]
    if mask_path.suffix.lower() in (".tif", ".tiff"):
        with rasterio.open(mask_path) as src:
            target = src.read(1)
            valid = src.read_masks(1) > 0
    else:
        with Image.open(mask_path) as src:
            target = np.asarray(src.convert("L"))
        valid = np.ones(target.shape, dtype=bool)
    update_counts(recounted, prediction > 0, target > 0, valid=valid)
for key in ("tp", "fp", "fn", "tn"):
    assert recounted[key] == summary["oof"]["micro"]["class_1"][key]
manifest = json.loads((output_dir / "kfold_manifest.json").read_text(encoding="utf-8"))
expected = [
    path
    for i in summary["folds_completed"]
    for path in manifest["fold_assignment"][f"fold_{i}"]["val_samples"]
]
assert sorted(str(p.resolve()) for p in prediction_lookup) == sorted(expected)
print("PASS: held-out coverage and pooled counts verified from saved masks.")


## Visualize results

In [ ]:
test_image_path = next(iter(prediction_lookup))
ground_truth_path = mask_by_stem[test_image_path.stem]
prediction_path = prediction_lookup[test_image_path]
save_path = output_dir / f"{test_image_path.stem}_oof_comparison.png"
fig = geoai.plot_prediction_comparison(
    original_image=str(test_image_path),
    prediction_image=str(prediction_path),
    ground_truth_image=str(ground_truth_path),
    titles=["Original", "Held-out prediction", "Ground Truth"],
    figsize=(15, 5),
    save_path=str(save_path),
    show_plot=True,
    indexes=[1, 2, 3],
)


## Run ensemble inference

`monitor="val_loss"` selects each fold's minimum validation-loss checkpoint;
`monitor="val_iou"` selects its maximum validation-IoU checkpoint. The same
choice controls early stopping. `best_model.pth` contains that chosen model and
`model_selection.json` records the criterion, score, and selected epoch.

For inference, pass all selected fold checkpoints to
`geoai.semantic_segmentation_batch`. It averages **class probabilities** with
uniform weights, then applies argmax or the chosen probability threshold.
All folds must be complete for the full CV ensemble.

Set `ENSEMBLE_INPUT_DIR` to new imagery for deployment. Its configured default uses
this example's images to demonstrate ensemble predictions. Those predictions
are **not OOF results** because other ensemble members trained on each image.
Keep the per-fold OOF metrics above for CV assessment; evaluate the full ensemble
on an independent test dataset for an unbiased ensemble score.


In [ ]:
ensemble_predictions_dir = output_dir / "ensemble_predictions"
if summary["complete"]:
    geoai.semantic_segmentation_batch(
        input_dir=str(ENSEMBLE_INPUT_DIR),
        output_dir=str(ensemble_predictions_dir),
        model_path=summary["ensemble_model_paths"],
        architecture="unet",
        encoder_name=ENCODER_NAME,
        num_channels=3,
        num_classes=2,
        window_size=512,
        overlap=256,
        batch_size=BATCH_SIZE,
        device=device,
        quiet=True,
        raise_on_error=True,
    )
    print("Ensemble checkpoint criterion:", summary["monitor"])
    print("Ensemble models:", len(summary["ensemble_model_paths"]))
    print("Ensemble predictions:", ensemble_predictions_dir)
else:
    print("Skipping the full ensemble until every fold has completed.")


In [ ]:
if summary["complete"]:
    ensemble_image = next(
        p
        for p in sorted(ENSEMBLE_INPUT_DIR.iterdir())
        if p.suffix.lower() in (".tif", ".tiff", ".png", ".jpg", ".jpeg")
    )
    extension = ".tif" if ensemble_image.suffix.lower() in (".tif", ".tiff") else ".png"
    geoai.plot_prediction_comparison(
        original_image=str(ensemble_image),
        prediction_image=str(
            ensemble_predictions_dir / f"{ensemble_image.stem}_mask{extension}"
        ),
        titles=["Original", "Ensemble prediction"],
        figsize=(10, 5),
        show_plot=True,
        indexes=[1, 2, 3],
    )


## Re-run a single fold

Use the same settings, set `only_fold` to a zero-based fold index, and pass
`overwrite=True` to the training function. Other compatible completed results
are retained. Use a new experiment directory when changing data or settings.
